In [ ]:
import random
import uuid
import string
import json
import time
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import parse_json, col, lit
from snowflake.snowpark.types import StructType, StructField, StringType

session = get_active_session()

In [ ]:
CREATE OR REPLACE DATABASE STREAM;
CREATE OR REPLACE  SCHEMA STREAM.PUBLIC;

In [ ]:
CREATE OR REPLACE ICEBERG TABLE STREAM.PUBLIC.STREAM1 (
    DATA VARIANT
)
    CATALOG = 'SNOWFLAKE'
    EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
    ICEBERG_VERSION = 3;

In [ ]:
COUNTRIES = ["US", "FR", "DE", "GB", "JP", "BR", "IN", "CA", "AU", "MX"]
UTM_CAMPAIGNS = ["retargeting", "brand_awareness", "conversion", "loyalty", "seasonal"]
UTM_MEDIUMS = ["social", "email", "cpc", "organic", "referral"]
UTM_SOURCES = ["email", "facebook", "google", "twitter", "linkedin", "instagram"]
VARIANTS = ["control", "variant_a", "variant_b", "variant_c"]
BROWSERS = ["Chrome", "Firefox", "Safari", "Edge", "Opera"]
DEVICE_TYPES = ["desktop", "mobile", "tablet"]
OS_LIST = [
    "Windows 10", "Windows 11", "Windows Server 2022",
    "macOS 12", "macOS 13", "macOS 14", "macOS 15",
    "Ubuntu 20.04", "Ubuntu 22.04", "Ubuntu 24.04",
    "iOS 16", "iOS 17", "iOS 18",
    "Android 12", "Android 13", "Android 14", "Android 15",
    "Fedora 39", "Fedora 40",
    "Debian 11", "Debian 12",
    "ChromeOS 120", "ChromeOS 125"
]
RESOLUTIONS = ["1920x1080", "1366x768", "2560x1440", "1440x900", "375x812", "390x844"]
EVENT_TYPES = ["page_view", "click", "form_submit", "scroll", "purchase", "add_to_cart", "sign_up"]
TAGS_POOL = ["trending", "recommended", "premium", "new", "featured", "popular", "sale", "limited"]

def generate_record():
    record = {
        "custom_properties": {
            "country": random.choice(COUNTRIES),
            "experiment_id": f"exp_{uuid.uuid4().hex[:6]}",
            "utm_campaign": random.choice(UTM_CAMPAIGNS),
            "utm_medium": random.choice(UTM_MEDIUMS),
            "utm_source": random.choice(UTM_SOURCES),
            "variant": random.choice(VARIANTS)
        },
        "data": uuid.uuid4().hex * 4,
        "device_context": {
            "browser": random.choice(BROWSERS),
            "browser_version": f"{random.randint(100, 125)}.0",
            "device_type": random.choice(DEVICE_TYPES),
            "os": random.choice(OS_LIST),
            "screen_resolution": random.choice(RESOLUTIONS),
            "user_agent": "Mozilla/5.0 (compatible; StreamGen/1.0)"
        },
        "event_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
        "event_type": random.choice(EVENT_TYPES),
        "page_url": f"https://example.com/app/{uuid.uuid4().hex[:16]}",
        "session_id": str(uuid.uuid4()),
        "tags": random.sample(TAGS_POOL, k=random.randint(1, 4)),
        "user_id": f"user_{uuid.uuid4().hex[:12]}"
    }
    return json.dumps(record)

In [ ]:
click_pct = 70
add_to_cart_pct = 25
purchase_pct = 5
row_count = 100_000_000
other_events = ['page_view', 'form_submit', 'scroll', 'sign_up']
other_pct_each = (100 - click_pct - add_to_cart_pct - purchase_pct) // len(other_events)

In [ ]:
ALTER WAREHOUSE MYWH SET WAREHOUSE_SIZE = 'LARGE';

INSERT INTO STREAM.PUBLIC.STREAM1 (DATA)
WITH vals AS (
  SELECT
    ABS(MOD(RANDOM(), 10)) AS r0,
    ABS(MOD(RANDOM(), 5)) AS r1,
    ABS(MOD(RANDOM(), 5)) AS r2,
    ABS(MOD(RANDOM(), 6)) AS r3,
    ABS(MOD(RANDOM(), 4)) AS r4,
    ABS(MOD(RANDOM(), 5)) AS r5,
    ABS(MOD(RANDOM(), 26)) + 100 AS r6,
    ABS(MOD(RANDOM(), 3)) AS r7,
    ABS(MOD(RANDOM(), 13)) AS r8,
    ABS(MOD(RANDOM(), 6)) AS r9,
    ABS(MOD(RANDOM(), 100)) AS r_event,
    ABS(MOD(RANDOM(), 86400)) AS r11,
    ABS(MOD(RANDOM(), 5)) AS r12,
    ABS(MOD(RANDOM(), 3)) AS r13,
    MD5(RANDOM()::VARCHAR) AS id1,
    MD5(RANDOM()::VARCHAR) AS id2
  FROM TABLE(GENERATOR(ROWCOUNT => {{row_count}}))
)
SELECT OBJECT_CONSTRUCT(
  'custom_properties', OBJECT_CONSTRUCT(
    'country', ARRAY_CONSTRUCT('US','FR','DE','GB','JP','BR','IN','CA','AU','MX')[r0],
    'experiment_id', 'exp_' || SUBSTR(id1, 1, 6),
    'utm_campaign', ARRAY_CONSTRUCT('retargeting','brand_awareness','conversion','loyalty','seasonal')[r1],
    'utm_medium', ARRAY_CONSTRUCT('social','email','cpc','organic','referral')[r2],
    'utm_source', ARRAY_CONSTRUCT('email','facebook','google','twitter','linkedin','instagram')[r3],
    'variant', ARRAY_CONSTRUCT('control','variant_a','variant_b','variant_c')[r4]
  ),
  'data', id1 || id2 || id1 || id2,
  'device_context', OBJECT_CONSTRUCT(
    'browser', ARRAY_CONSTRUCT('Chrome','Firefox','Safari','Edge','Opera')[r5],
    'browser_version', r6::VARCHAR || '.0',
    'device_type', ARRAY_CONSTRUCT('desktop','mobile','tablet')[r7],
    'os', ARRAY_CONSTRUCT('Windows 10','Windows 11','macOS 13','macOS 14','Ubuntu 22.04','Ubuntu 24.04','iOS 16','iOS 17','iOS 18','Android 12','Android 13','Android 14','ChromeOS 120')[r8],
    'screen_resolution', ARRAY_CONSTRUCT('1920x1080','1366x768','2560x1440','1440x900','375x812','390x844')[r9],
    'user_agent', 'Mozilla/5.0 (compatible; StreamGen/1.0)'
  ),
  'event_timestamp', TO_VARCHAR(DATEADD('second', -r11, CURRENT_TIMESTAMP()), 'YYYY-MM-DD"T"HH24:MI:SS.FF6"Z"'),
  'event_type', CASE
    WHEN r_event < {{click_pct}} THEN 'click'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} THEN 'add_to_cart'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} THEN 'purchase'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} + {{other_pct_each}} THEN 'page_view'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} + {{other_pct_each}} * 2 THEN 'form_submit'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} + {{other_pct_each}} * 3 THEN 'scroll'
    ELSE 'sign_up'
  END,
  'page_url', 'https://example.com/app/' || SUBSTR(id2, 1, 16),
  'session_id', SUBSTR(id1, 1, 8) || '-' || SUBSTR(id1, 9, 4) || '-' || SUBSTR(id1, 13, 4) || '-' || SUBSTR(id2, 1, 4) || '-' || SUBSTR(id2, 5, 12),
  'tags', ARRAY_SLICE(
    ARRAY_CONSTRUCT('trending','recommended','premium','new','featured','popular','sale','limited'),
    r12, r12 + 1 + r13
  ),
  'user_id', 'user_' || SUBSTR(id1, 21, 12)
) AS DATA
FROM vals;

ALTER WAREHOUSE MYWH SET WAREHOUSE_SIZE = 'X-SMALL';

In [ ]:
ALTER WAREHOUSE MYWH SET WAREHOUSE_SIZE = 'MEDIUM';

In [ ]:
import multiprocessing.pool
import itertools

BATCH_SIZE = 100000
NUM_ITERATIONS = 1
NUM_WORKERS = 32

schema = StructType([StructField("RAW_JSON", StringType())])

def generate_batch(n):
    return [[generate_record()] for _ in range(n)]

chunk_size = BATCH_SIZE // NUM_WORKERS

for i in range(NUM_ITERATIONS):
    with multiprocessing.pool.ThreadPool(NUM_WORKERS) as pool:
        chunks = pool.map(generate_batch, [chunk_size] * NUM_WORKERS)
    records = list(itertools.chain.from_iterable(chunks))
    df = session.create_dataframe(records, schema=schema)
    df_variant = df.select(parse_json(col("RAW_JSON")).alias("DATA"))
    df_variant.write.mode("append").save_as_table("STREAM.PUBLIC.STREAM1", iceberg_config={})
    print(f"Batch {i + 1}/{NUM_ITERATIONS}: Inserted {len(records)} records")

print(f"Done. Total records inserted: {BATCH_SIZE * NUM_ITERATIONS}")

## Explore STREAM.PUBLIC.STREAM1

In [ ]:
%%sql -r sample_data
SELECT DATA FROM STREAM.PUBLIC.STREAM1 LIMIT 5;

In [ ]:
%%sql -r overview_stats
SELECT
  COUNT(*) AS row_count,
  MIN(DATA:event_timestamp::TIMESTAMP) AS earliest_event,
  MAX(DATA:event_timestamp::TIMESTAMP) AS latest_event,
  COUNT(DISTINCT DATA:user_id::STRING) AS unique_users,
  COUNT(DISTINCT DATA:session_id::STRING) AS unique_sessions
FROM STREAM.PUBLIC.STREAM1

In [ ]:
%%sql -r event_distribution
SELECT
  DATA:event_type::STRING AS event_type,
  COUNT(*) AS cnt,
  COUNT(DISTINCT DATA:user_id::STRING) AS unique_users
FROM STREAM.PUBLIC.STREAM1
GROUP BY 1
ORDER BY 2 DESC

In [ ]:
%%sql -r device_geo_breakdown
SELECT
  DATA:device_context.device_type::STRING AS device_type,
  DATA:custom_properties.country::STRING AS country,
  COUNT(*) AS cnt
FROM STREAM.PUBLIC.STREAM1
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 15

In [ ]:
%%sql -r campaign_breakdown
SELECT
  DATA:custom_properties.utm_source::STRING AS utm_source,
  DATA:custom_properties.utm_campaign::STRING AS utm_campaign,
  COUNT(*) AS cnt
FROM STREAM.PUBLIC.STREAM1
GROUP BY 1, 2
ORDER BY 3 DESC

## User Behavior Funnel Analysis
Tracking event sequences: click → add_to_cart → purchase to identify drop-off points.

In [ ]:
%%sql -r funnel_overview
WITH user_events AS (
  SELECT
    DATA:user_id::STRING AS user_id,
    DATA:event_type::STRING AS event_type,
    DATA:event_timestamp::TIMESTAMP AS event_ts
  FROM STREAM.PUBLIC.STREAM1
  WHERE DATA:event_type::STRING IN ('click', 'add_to_cart', 'purchase')
),
funnel AS (
  SELECT
    user_id,
    MAX(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) AS had_click,
    MAX(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS had_add_to_cart,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS had_purchase
  FROM user_events
  GROUP BY user_id
)
SELECT
  'click' AS funnel_step,
  1 AS step_order,
  COUNT_IF(had_click = 1) AS users,
  ROUND(COUNT_IF(had_click = 1) * 100.0 / COUNT(*), 1) AS pct_of_total
FROM funnel
UNION ALL
SELECT
  'add_to_cart',
  2,
  COUNT_IF(had_add_to_cart = 1),
  ROUND(COUNT_IF(had_add_to_cart = 1) * 100.0 / COUNT(*), 1)
FROM funnel
UNION ALL
SELECT
  'purchase',
  3,
  COUNT_IF(had_purchase = 1),
  ROUND(COUNT_IF(had_purchase = 1) * 100.0 / COUNT(*), 1)
FROM funnel
ORDER BY step_order

In [ ]:
%%sql -r funnel_dropoff
WITH user_events AS (
  SELECT
    DATA:user_id::STRING AS user_id,
    DATA:event_type::STRING AS event_type,
    DATA:event_timestamp::TIMESTAMP AS event_ts
  FROM STREAM.PUBLIC.STREAM1
  WHERE DATA:event_type::STRING IN ('click', 'add_to_cart', 'purchase')
),
funnel AS (
  SELECT
    user_id,
    MAX(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) AS had_click,
    MAX(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS had_add_to_cart,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS had_purchase
  FROM user_events
  GROUP BY user_id
)
SELECT
  'click → add_to_cart' AS transition,
  COUNT_IF(had_click = 1) AS from_step_users,
  COUNT_IF(had_click = 1 AND had_add_to_cart = 1) AS to_step_users,
  ROUND(COUNT_IF(had_click = 1 AND had_add_to_cart = 1) * 100.0 / NULLIF(COUNT_IF(had_click = 1), 0), 1) AS conversion_rate_pct,
  ROUND((1 - COUNT_IF(had_click = 1 AND had_add_to_cart = 1) * 1.0 / NULLIF(COUNT_IF(had_click = 1), 0)) * 100, 1) AS drop_off_rate_pct
FROM funnel
UNION ALL
SELECT
  'add_to_cart → purchase',
  COUNT_IF(had_add_to_cart = 1),
  COUNT_IF(had_add_to_cart = 1 AND had_purchase = 1),
  ROUND(COUNT_IF(had_add_to_cart = 1 AND had_purchase = 1) * 100.0 / NULLIF(COUNT_IF(had_add_to_cart = 1), 0), 1),
  ROUND((1 - COUNT_IF(had_add_to_cart = 1 AND had_purchase = 1) * 1.0 / NULLIF(COUNT_IF(had_add_to_cart = 1), 0)) * 100, 1)
FROM funnel

In [ ]:
%%sql -r funnel_by_segment
WITH user_events AS (
  SELECT
    DATA:user_id::STRING AS user_id,
    DATA:event_type::STRING AS event_type,
    DATA:custom_properties.country::STRING AS country,
    DATA:device_context.device_type::STRING AS device_type
  FROM STREAM.PUBLIC.STREAM1
  WHERE DATA:event_type::STRING IN ('click', 'add_to_cart', 'purchase')
),
funnel_by_segment AS (
  SELECT
    country,
    device_type,
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN event_type = 'click' THEN user_id END) AS clicks,
    COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS add_to_carts,
    COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS purchases
  FROM user_events
  GROUP BY 1, 2
)
SELECT
  country,
  device_type,
  total_users,
  clicks,
  add_to_carts,
  purchases,
  ROUND(add_to_carts * 100.0 / NULLIF(clicks, 0), 1) AS click_to_cart_pct,
  ROUND(purchases * 100.0 / NULLIF(add_to_carts, 0), 1) AS cart_to_purchase_pct,
  ROUND(purchases * 100.0 / NULLIF(clicks, 0), 1) AS overall_conversion_pct
FROM funnel_by_segment
ORDER BY total_users DESC
LIMIT 20

In [ ]:


click_pct = 70
add_to_cart_pct = 25
purchase_pct = 5

other_events = ['page_view', 'form_submit', 'scroll', 'sign_up']
other_pct_each = (100 - click_pct - add_to_cart_pct - purchase_pct) // len(other_events)